# Laya Studio: label, fine-tune and test Laya on Kaggle, behind a Cloudflare Tunnel

A web UI on port **7860** with five tabs:

| Tab | What it does |
|---|---|
| **Dataset** | Upload JSONL/CSV, import files attached under `/kaggle/input`, browse/search/delete records, see label balance |
| **Label** | Paste a bill/receipt/transaction, answer the template questions (optionally pre-filled by a model), save |
| **Train** | Fine-tune `laya`, `laya-typed-decisions`, `laya-multilingual` or one of your earlier runs; live progress and before/after accuracy |
| **Test** | Run any base or fine-tuned model on a document and see each decision with its probabilities |
| **Models** | Download a run as .zip, push it to Hugging Face, delete runs |

**What Laya does:** typed decisions over text or JSON: *choice* (which category), *noul* (yes/no) and *score* (ordered level), in one forward pass.
It does **not** read images or extract values such as totals or dates. For scanned bills, run OCR first and feed the text (plus any fields you already have) as the document.

### Before you run
1. **Accelerator:** GPU T4 x2 (one GPU also works). **Internet:** on.
2. **A new Cloudflare Tunnel** for this notebook, e.g. `laya.example.com` → `http://localhost:7860`.
   Do not reuse the Ollama notebook's tunnel: two notebooks on one tunnel would randomly split traffic between them.
3. **Kaggle secrets** (Add-ons → Secrets):
   - `LAYA_TUNNEL_TOKEN`: the new tunnel's token (required)
   - `LAYA_UI_PASSWORD`: password for the UI (recommended; if missing, a random one is printed below)
   - `HF_TOKEN`: Hugging Face token with write access (optional, for "Push to HF")
4. Run All. The last cell keeps the session alive; the UI keeps working while it runs.

`/kaggle/working` is wiped when the session ends. Keep your dataset by downloading it (Dataset tab), or attach it as a Kaggle dataset and use **Import**; keep models via Download or Push to HF.

In [ ]:
# 1. Install Laya and cloudflared (torch/transformers are already on Kaggle)
!pip install -q laya || pip install -q git+https://github.com/NandhaKishorM/laya
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /kaggle/working/cloudflared && chmod +x /kaggle/working/cloudflared
!/kaggle/working/cloudflared --version
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: enable a GPU accelerator to train"
import laya, torch, transformers
print("laya", getattr(laya, "__version__", "?"), "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPUs:", torch.cuda.device_count())

### 2. App files
These cells only write files; nothing runs yet.

In [ ]:
# Dataset helpers and the bills / receipts / fintech question templates
# Writes laya_data.py into APP_DIR. Safe to re-run.
import os
APP_DIR = "/kaggle/working/laya_app"
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'laya_data.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Dataset helpers for Laya Studio (pure Python, no torch needed).

A record is one document plus the decisions labelled for it:

    {"state": {"text": "..."},                      # any text / JSON the model reads
     "questions": {"doc_type": {"type": "choice", "instructions": "...", "criteria": {...}}, ...},
     "gold": {"doc_type": "receipt", "vat_charged": true, "risk_level": 2}}

`gold` values may be a plain label (as above) or a dict with "label" and/or "probabilities",
which is the format of the official LocalLLaMA/typed-decisions dataset.
"""
import copy
import csv
import io
import json

QTYPE_NAMES = ("choice", "score", "noul")

# ---------------------------------------------------------------- starter templates
TEMPLATES = {
    "receipts_bills": {
        "title": "Bills & receipts",
        "questions": {
            "doc_type": {
                "type": "choice",
                "instructions": "What kind of financial document is this?",
                "criteria": {
                    "invoice": "a bill requesting payment, usually with an invoice number and due date",
                    "receipt": "proof that a payment was already made at a shop, restaurant or online",
                    "utility_bill": "electricity, water, internet, phone or gas bill",
                    "bank_statement": "a list of account transactions issued by a bank or wallet",
                    "payslip": "salary or wage statement",
                    "tax_document": "VAT/PAN certificate, tax return or tax receipt",
                    "other": "none of the above",
                },
            },
            "expense_category": {
                "type": "choice",
                "instructions": "Which expense category does this document belong to?",
                "criteria": {
                    "food_dining": "restaurants, cafes, food delivery",
                    "groceries": "supermarkets, grocery and household goods",
                    "transport": "fuel, taxi, ride-hailing, bus, parking, vehicle service",
                    "utilities": "electricity, water, internet, phone",
                    "rent": "office or home rent, lease",
                    "office_supplies": "stationery, equipment, printing",
                    "software": "software, SaaS, cloud or domain subscriptions",
                    "travel": "flights, hotels, tours",
                    "medical": "hospital, clinic, pharmacy",
                    "professional_services": "legal, accounting, consulting, contractors",
                    "other": "none of the above",
                },
            },
            "payment_method": {
                "type": "choice",
                "instructions": "How was this paid, or how is it to be paid?",
                "criteria": {
                    "cash": "cash payment",
                    "card": "debit or credit card",
                    "bank_transfer": "bank transfer, Fonepay/QR to a bank account, or deposit",
                    "mobile_wallet": "eSewa, Khalti, IME Pay or another wallet",
                    "cheque": "cheque",
                    "unknown": "not stated",
                },
            },
            "vat_charged": {
                "type": "noul",
                "instructions": "Does the document show VAT or another tax being charged?",
            },
            "needs_review": {
                "type": "noul",
                "instructions": "Is information missing, unreadable, duplicated or inconsistent "
                                "(for example totals that do not add up) so a person should review it?",
            },
        },
    },
    "fintech_transactions": {
        "title": "Fintech transactions",
        "questions": {
            "txn_type": {
                "type": "choice",
                "instructions": "What type of transaction is this?",
                "criteria": {
                    "payment": "payment to a merchant or biller",
                    "refund": "money returned for an earlier payment",
                    "transfer": "person-to-person or account-to-account transfer",
                    "withdrawal": "cash withdrawal or cash-out",
                    "deposit": "cash-in, top-up or deposit",
                    "fee": "service charge, commission or penalty",
                    "salary": "salary or payroll credit",
                    "loan_repayment": "loan or EMI repayment",
                    "other": "none of the above",
                },
            },
            "is_suspicious": {
                "type": "noul",
                "instructions": "Does this transaction look suspicious or potentially fraudulent?",
            },
            "risk_level": {
                "type": "score",
                "instructions": "How risky is this transaction?",
                "criteria": [
                    "low: routine, consistent with normal activity",
                    "moderate: slightly unusual amount, time or counterparty",
                    "elevated: several unusual signals, should be checked",
                    "high: strong fraud or money-laundering signals, block or escalate",
                ],
            },
        },
    },
    "fintech_support": {
        "title": "Fintech customer support",
        "questions": {
            "intent": {
                "type": "choice",
                "instructions": "What does the customer want?",
                "criteria": {
                    "failed_payment": "a payment failed or is stuck/pending",
                    "duplicate_charge": "charged twice for the same thing",
                    "refund_request": "wants money back",
                    "account_access": "cannot log in, locked account, PIN or OTP problem",
                    "kyc_verification": "KYC, identity documents or verification",
                    "balance_or_statement": "asks about balance, statement or transaction history",
                    "card_issue": "card blocked, lost, declined or not received",
                    "other": "none of the above",
                },
            },
            "urgency": {
                "type": "score",
                "instructions": "How urgent is this request?",
                "criteria": [
                    "low: general question, no money at risk",
                    "normal: needs an answer today",
                    "high: money is stuck or the customer cannot transact",
                    "critical: money lost, fraud or account compromised right now",
                ],
            },
            "needs_human": {
                "type": "noul",
                "instructions": "Does this need a human agent rather than an automated reply?",
            },
        },
    },
}

_TRUE = {"true", "yes", "y", "1", "t", "ho", "हो"}
_FALSE = {"false", "no", "n", "0", "f", "haina", "होइन"}


# ---------------------------------------------------------------- questions
def normalize_question(qid, q):
    """Validate one question definition and return a clean copy."""
    if not isinstance(q, dict):
        raise ValueError("question %r must be an object" % qid)
    t = q.get("type")
    if t not in QTYPE_NAMES:
        raise ValueError("question %r: type must be one of %s" % (qid, ", ".join(QTYPE_NAMES)))
    out = {"type": t, "instructions": str(q.get("instructions") or qid)}
    crit = q.get("criteria")
    if t == "choice":
        if isinstance(crit, list):
            crit = {str(k): "" for k in crit}
        if not isinstance(crit, dict) or len(crit) < 2:
            raise ValueError("question %r: a choice needs criteria with at least 2 options" % qid)
        out["criteria"] = {str(k): ("" if v is None else v) for k, v in crit.items()}
    elif t == "score":
        if not isinstance(crit, list) or len(crit) < 2:
            raise ValueError("question %r: a score needs criteria as a list of at least 2 levels" % qid)
        out["criteria"] = [str(c) for c in crit]
    else:
        if crit:
            if not isinstance(crit, dict) or not set(crit) <= {"true", "false"}:
                raise ValueError("question %r: noul criteria may only have 'true'/'false' keys" % qid)
            out["criteria"] = dict(crit)
    return out


def option_keys(q):
    """Option names in the order Laya renders them (noul is always [false, true])."""
    t = q["type"]
    if t == "choice":
        return list(q["criteria"].keys())
    if t == "score":
        return [str(i) for i in range(len(q["criteria"]))]
    return ["false", "true"]


def label_index(q, value):
    """Turn a human label into an option index. Raises ValueError when it doesn't fit."""
    t = q["type"]
    if isinstance(value, str):
        value = value.strip()
    if t == "noul":
        if isinstance(value, bool):
            return int(value)
        s = str(value).strip().lower()
        if s in _TRUE:
            return 1
        if s in _FALSE:
            return 0
        raise ValueError("expected yes/no (true/false), got %r" % (value,))
    if t == "choice":
        keys = option_keys(q)
        s = str(value)
        if s in keys:
            return keys.index(s)
        low = [k.lower() for k in keys]
        if s.lower() in low:
            return low.index(s.lower())
        raise ValueError("%r is not one of: %s" % (value, ", ".join(keys)))
    # score
    n = len(q["criteria"])
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        i = int(round(value))
    else:
        s = str(value).strip()
        try:
            i = int(round(float(s)))
        except ValueError:
            names = [str(c).split(":")[0].strip().lower() for c in q["criteria"]]
            if s.lower() in names:
                i = names.index(s.lower())
            else:
                raise ValueError("%r is not a level number 0-%d or one of: %s" % (value, n - 1, ", ".join(names)))
    if not 0 <= i < n:
        raise ValueError("level %d is outside 0-%d" % (i, n - 1))
    return i


def gold_to_target(q, g):
    """Return (target distribution over options, label index) for one gold answer."""
    keys = option_keys(q)
    k = len(keys)
    probs = None
    label = g
    if isinstance(g, dict):
        probs = g.get("probabilities")
        label = g.get("label", g.get("choice", g.get("score", g.get("noul"))))
    if isinstance(probs, dict) and probs:
        target = [max(0.0, float(probs.get(key, 0.0))) for key in keys]
    elif q["type"] == "noul" and isinstance(label, float) and 0.0 < label < 1.0:
        target = [1.0 - label, label]          # soft yes/no probability
    else:
        if label is None:
            raise ValueError("no label given")
        target = [0.0] * k
        target[label_index(q, label)] = 1.0
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / k] * k
    return target, max(range(k), key=lambda i: target[i])


# ---------------------------------------------------------------- records
def normalize_record(rec):
    """Validate a record; returns a clean copy (only questions that have a gold answer)."""
    if not isinstance(rec, dict):
        raise ValueError("record must be a JSON object")
    state, questions, gold = rec.get("state"), rec.get("questions"), rec.get("gold")
    # The official dataset stores these three as JSON strings.
    if isinstance(state, str) and state.strip()[:1] in "{[":
        try:
            state = json.loads(state)
        except ValueError:
            pass
    if isinstance(questions, str):
        questions = json.loads(questions)
    if isinstance(gold, str):
        gold = json.loads(gold)
    if state in (None, "", {}, []):
        raise ValueError("missing 'state' (the document text or JSON)")
    if not isinstance(questions, dict) or not isinstance(gold, dict):
        raise ValueError("'questions' and 'gold' must be objects")
    out_q, out_g = {}, {}
    for qid, g in gold.items():
        if qid not in questions:
            continue
        q = normalize_question(qid, questions[qid])
        try:
            gold_to_target(q, g)
        except ValueError as e:
            raise ValueError("%s: %s" % (qid, e))
        out_q[qid], out_g[qid] = q, g
    if not out_g:
        raise ValueError("no labelled answers (gold) that match a question")
    return {"state": state, "questions": out_q, "gold": out_g}


def parse_jsonl(text):
    records, errors = [], []
    for n, line in enumerate(text.splitlines(), 1):
        line = line.strip()
        if not line:
            continue
        try:
            records.append(normalize_record(json.loads(line)))
        except (ValueError, TypeError) as e:
            errors.append("line %d: %s" % (n, e))
    return records, errors


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        records, errors = parse_jsonl(f.read())
    return records


def parse_csv(text, questions):
    """CSV import. Columns named after question ids hold labels; every other column is the document.

    A single 'text' column becomes {"text": ...}; several columns become a JSON state such as
    {"merchant": ..., "amount": ..., "text": ...}. Empty label cells are skipped.
    """
    questions = {qid: normalize_question(qid, q) for qid, q in questions.items()}
    reader = csv.DictReader(io.StringIO(text.lstrip("﻿")))
    if not reader.fieldnames:
        return [], ["CSV has no header row"]
    fields = [f.strip() for f in reader.fieldnames if f is not None]
    qcols = [f for f in fields if f in questions]
    scols = [f for f in fields if f not in questions]
    if not qcols:
        return [], ["no column matches a question id of this template (%s)" % ", ".join(questions)]
    if not scols:
        return [], ["no document column found - add a 'text' column"]
    records, errors = [], []
    for n, row in enumerate(reader, 2):
        row = {(k or "").strip(): (v or "").strip() for k, v in row.items() if k is not None}
        state = {c: row.get(c, "") for c in scols if row.get(c, "")}
        if not state:
            errors.append("row %d: empty document" % n)
            continue
        gold, qs, bad = {}, {}, None
        for c in qcols:
            v = row.get(c, "")
            if v == "":
                continue
            try:
                label_index(questions[c], v)
            except ValueError as e:
                bad = "row %d, %s: %s" % (n, c, e)
                break
            gold[c], qs[c] = v, questions[c]
        if bad:
            errors.append(bad)
            continue
        if not gold:
            errors.append("row %d: no labels" % n)
            continue
        records.append({"state": state, "questions": qs, "gold": gold})
    return records, errors


def csv_example(questions):
    """A tiny example CSV for the help text."""
    qids = list(questions)
    row = []
    for qid in qids:
        q = normalize_question(qid, questions[qid])
        row.append(option_keys(q)[0] if q["type"] == "choice" else "yes" if q["type"] == "noul" else "1")
    return "text," + ",".join(qids) + "\n\"<document text>\"," + ",".join(row) + "\n"


def state_preview(state, n=160):
    s = state if isinstance(state, str) else (state.get("text") if isinstance(state, dict) and
                                               isinstance(state.get("text"), str) else json.dumps(state, ensure_ascii=False))
    s = " ".join(str(s).split())
    return s if len(s) <= n else s[: n - 1] + "…"


def gold_display(q, g):
    target, idx = gold_to_target(q, g)
    if q["type"] == "choice":
        return option_keys(q)[idx]
    if q["type"] == "noul":
        return "yes" if idx == 1 else "no"
    return "level %d" % idx


def dataset_stats(records):
    per_q = {}
    for rec in records:
        for qid, g in rec["gold"].items():
            q = rec["questions"][qid]
            d = per_q.setdefault(qid, {"type": q["type"], "counts": {}})
            try:
                lab = gold_display(q, g)
            except ValueError:
                continue
            d["counts"][lab] = d["counts"].get(lab, 0) + 1
    return {"records": len(records),
            "decisions": sum(len(r["gold"]) for r in records),
            "questions": per_q}


def default_templates():
    return copy.deepcopy(TEMPLATES)


def validate_templates(t):
    if not isinstance(t, dict) or not t:
        raise ValueError("templates must be a non-empty object")
    out = {}
    for tid, tpl in t.items():
        if not isinstance(tpl, dict) or not isinstance(tpl.get("questions"), dict) or not tpl["questions"]:
            raise ValueError("template %r needs a 'questions' object" % tid)
        out[str(tid)] = {"title": str(tpl.get("title") or tid),
                         "questions": {qid: normalize_question(qid, q) for qid, q in tpl["questions"].items()}}
    return out
''')
print("wrote", os.path.join(APP_DIR, 'laya_data.py'))

In [ ]:
# Fine-tuning script launched by the UI (based on the official Laya Kaggle notebook)
# Writes train_laya.py into APP_DIR. Safe to re-run.
import os
APP_DIR = "/kaggle/working/laya_app"
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'train_laya.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Fine-tune a Laya checkpoint on a Laya Studio dataset.

Launched by the Studio UI as:  torchrun --standalone --nproc_per_node=N train_laya.py job.json
(or plain `python train_laya.py job.json` without a GPU).

The loss is the one from the official Laya Kaggle notebook
(notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb): a policy-gradient term on a
proper scoring reward plus soft cross-entropy, followed by per-type temperature calibration.
Additions here: 1..N GPUs, a document-level validation split, baseline/per-epoch accuracy,
a status.json the UI polls, and a load-and-predict smoke test of the saved model.
"""
import datetime
import json
import math
import os
import random
import sys
import time
import traceback

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import torch
import torch.distributed as dist

from laya_data import gold_to_target, load_jsonl

HUB_FILES = ["rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*"]
FWD_KEYS = ("input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype")


def collate(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {"input_ids": ids, "attention_mask": att, "marker_pos": mpos, "marker_mask": mmask,
            "target": target, "qtype": torch.tensor([it["qtype"] for it in items]),
            "label": torch.tensor([it["label"] for it in items])}


def fit_one_temp(sel):
    """Temperature that minimises cross-entropy of softmax(logits / T) against the targets."""
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, : len(z)] = torch.tensor(z)
        T[i, : len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss

    opt.step(closure)
    t = float(log_t.exp().item())
    # laya clamps temperatures to [0.5, 5.0] at load time and warns outside it
    return 1.0 if not math.isfinite(t) else min(5.0, max(0.5, t))


class Status:
    def __init__(self, path, enabled):
        self.path, self.enabled = path, enabled
        self.d = {"phase": "starting", "started": time.time()}

    def set(self, **kw):
        if not self.enabled:
            return
        self.d.update(kw)
        self.d["updated"] = time.time()
        tmp = self.path + ".tmp"
        with open(tmp, "w") as f:
            json.dump(self.d, f)
        os.replace(tmp, self.path)


def evaluate(model, items, pad_id, device, use_amp, bs=16):
    """Accuracy (argmax == gold label) overall and per question id; also returns logits."""
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(items), bs):
            chunk = items[i: i + bs]
            b = collate(chunk, pad_id)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                logits, _ = model(*(b[k].to(device) for k in FWD_KEYS))
            l = logits.float().cpu()
            for r, it in enumerate(chunk):
                preds.append((it, l[r, : len(it["markers"])].tolist()))
    per_q, correct = {}, 0
    for it, z in preds:
        ok = int(max(range(len(z)), key=lambda j: z[j]) == it["label"])
        correct += ok
        d = per_q.setdefault(it["qid"], [0, 0])
        d[0] += ok
        d[1] += 1
    metrics = {"accuracy": round(correct / len(preds), 4) if preds else None, "n": len(preds),
               "per_question": {q: {"accuracy": round(c / n, 4), "n": n} for q, (c, n) in sorted(per_q.items())}}
    return metrics, preds


def main():
    job = json.load(open(sys.argv[1]))
    run_dir = job["run_dir"]
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    rank = int(os.environ.get("RANK", "0"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    use_cuda = torch.cuda.is_available()
    distributed = world_size > 1
    if distributed:
        dist.init_process_group("nccl" if use_cuda else "gloo", timeout=datetime.timedelta(hours=3))
    if use_cuda:
        torch.cuda.set_device(local_rank)
        device = torch.device("cuda", local_rank)
    else:
        device = torch.device("cpu")
    is_main = rank == 0
    st = Status(os.path.join(run_dir, "status.json"), is_main)
    try:
        train(job, run_dir, world_size, rank, local_rank, device, use_cuda, distributed, is_main, st)
    except Exception:
        st.set(phase="error", error=traceback.format_exc()[-4000:])
        raise
    finally:
        if distributed:
            dist.destroy_process_group()


def train(job, run_dir, world_size, rank, local_rank, device, use_cuda, distributed, is_main, st):
    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file, save_file
    from transformers import AutoTokenizer
    from laya.agent import _fix_tokenizer_config
    from laya.common import QTYPES, build_model, build_sequence, proper_reward

    def log(*a):
        if is_main:
            print(*a, flush=True)

    def barrier():
        if distributed:
            dist.barrier()

    t_start = time.time()
    base = job["base"]

    # ------------------------------------------------------------ base checkpoint
    st.set(phase="downloading", message="Fetching %s" % base)

    def resolve():
        return base if os.path.isdir(base) else snapshot_download(base, allow_patterns=HUB_FILES)

    if is_main:
        model_dir = resolve()
        _fix_tokenizer_config(model_dir)
    barrier()
    if not is_main:
        model_dir = resolve()
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    max_len = int(job.get("max_len") or cfg.get("max_len", 512))
    head_max_len = int(cfg.get("head_max_len", 192))
    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    log("Base: %s | max_len=%d head_max_len=%d | world_size=%d | device=%s"
        % (base, max_len, head_max_len, world_size, device))

    # ------------------------------------------------------------ data
    st.set(phase="preparing", message="Tokenising dataset")
    records = load_jsonl(job["dataset"])
    order = list(range(len(records)))
    random.Random(int(job.get("seed", 42))).shuffle(order)
    val_frac = float(job.get("val_frac", 0.15))
    n_val = int(round(len(records) * val_frac)) if len(records) >= 10 and val_frac > 0 else 0
    val_ids = set(order[:n_val])       # split by document so no document leaks into both

    def build(ids):
        items, skipped = [], 0
        for i in ids:
            rec = records[i]
            for qid, g in rec["gold"].items():
                q = rec["questions"][qid]
                target, label = gold_to_target(q, g)
                iq = {"t": q["type"], "ins": q.get("instructions", ""), "crit": q.get("criteria")}
                seq, markers = build_sequence(tok, rec["state"], iq, max_len, head_max_len)
                if len(markers) != len(target):
                    skipped += 1       # options did not fit in head_max_len
                    continue
                items.append({"ids": seq, "markers": markers, "qtype": QTYPES[q["type"]],
                              "target": target, "label": label, "qid": qid})
        return items, skipped

    train_items, sk1 = build([i for i in order if i not in val_ids])
    val_items, sk2 = build([i for i in order if i in val_ids])
    per_rank = len(train_items) // world_size
    if per_rank == 0:
        raise RuntimeError("Not enough training decisions (%d) for %d GPU(s)" % (len(train_items), world_size))
    # equal-length shards: unequal counts make DDP ranks wait on each other forever
    my_items = train_items[rank::world_size][:per_rank]
    log("Records: %d (%d train / %d val) | decisions: %d train, %d val | skipped: %d"
        % (len(records), len(records) - n_val, n_val, len(train_items), len(val_items), sk1 + sk2))

    # ------------------------------------------------------------ model
    st.set(phase="loading_model", message="Loading weights")
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
    if use_cuda:
        model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.head_checkpointing = True
    model.to(device)
    use_amp = use_cuda
    pad_id = tok.pad_token_id

    baseline = None
    if is_main and val_items:
        st.set(phase="baseline_eval", message="Scoring the base model on the validation set")
        baseline, _ = evaluate(model, val_items, pad_id, device, use_amp)
        log("Baseline val accuracy: %s" % baseline["accuracy"])
    barrier()

    model.train()
    if distributed:
        from torch.nn.parallel import DistributedDataParallel as DDP
        net = DDP(model, device_ids=[local_rank] if use_cuda else None, find_unused_parameters=True)
    else:
        net = model

    EPOCHS = int(job.get("epochs", 4))
    MICRO_BATCH = int(job.get("micro_batch", 8))
    GRAD_ACCUM = int(job.get("grad_accum", 4))
    GROUP_SIZE = 4
    LR_ENCODER = float(job.get("lr_encoder", 2.5e-5))
    LR_HEAD = float(job.get("lr_head", 1.0e-4))
    SIGMA_START, SIGMA_END = 0.4, 0.1

    enc_params = [p for n, p in net.named_parameters() if "encoder." in n]
    head_params = [p for n, p in net.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENCODER},
                                   {"params": head_params, "lr": LR_HEAD}], weight_decay=0.01)
    micro_per_epoch = math.ceil(len(my_items) / MICRO_BATCH)
    total_updates = math.ceil(micro_per_epoch / GRAD_ACCUM) * EPOCHS
    total_micro = micro_per_epoch * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    hist, done_micro, t0 = [], 0, time.time()
    recent = []
    st.set(phase="training", epoch=0, epochs=EPOCHS, step=0, total_steps=total_micro, baseline=baseline,
           message="Training on %d decisions x %d GPU(s)" % (len(my_items) * world_size, world_size))
    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches, accum_step = 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * (epoch / max(1, EPOCHS - 1))
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx: b_idx + MICRO_BATCH]
            batch = collate(chunk, pad_id)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                logits, act = net(*(batch[k].to(device) for k in FWD_KEYS))
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            qtype = batch["qtype"].to(device)

            # G noisy logit samples (zero-mean over options) scored with a proper scoring rule
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), qtype, mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

            scaler.scale(loss).backward()
            accum_step += 1
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            lv = loss.item() * GRAD_ACCUM
            epoch_loss += lv
            n_batches += 1
            done_micro += 1
            recent = (recent + [lv])[-25:]
            if is_main and (done_micro % 10 == 0 or done_micro == total_micro):
                el = time.time() - t0
                st.set(epoch=epoch + 1, step=done_micro, loss=round(sum(recent) / len(recent), 4),
                       reward=round(r.mean().item(), 4), lr=scheduler.get_last_lr()[0],
                       eta_s=int(el / done_micro * (total_micro - done_micro)))
            if is_main and done_micro % 50 == 0:
                log("epoch %d/%d step %d/%d loss %.4f" % (epoch + 1, EPOCHS, done_micro, total_micro, lv))

        barrier()
        entry = {"epoch": epoch + 1, "train_loss": round(epoch_loss / max(1, n_batches), 4)}
        if is_main and val_items:
            st.set(message="Validating after epoch %d" % (epoch + 1))
            m, _ = evaluate(model, val_items, pad_id, device, use_amp)
            entry["val_accuracy"] = m["accuracy"]
            model.train()
        if is_main:
            hist.append(entry)
            st.set(history=hist, message="Epoch %d/%d done" % (epoch + 1, EPOCHS))
            log("=== epoch %d/%d | loss %.4f | val acc %s | %.0fs ===" % (
                epoch + 1, EPOCHS, entry["train_loss"], entry.get("val_accuracy"), time.time() - t0))
        barrier()

    if not is_main:
        return

    # ------------------------------------------------------------ calibrate, evaluate, save
    del optimizer, scaler, scheduler
    if use_cuda:
        torch.cuda.empty_cache()
    st.set(phase="calibrating", message="Fitting confidence temperatures")
    final, val_preds = evaluate(model, val_items, pad_id, device, use_amp) if val_items else (None, [])
    calib = val_preds if len(val_preds) >= 30 else evaluate(
        model, train_items[:: max(1, len(train_items) // 400)][:400], pad_id, device, use_amp)[1]
    temps = [1.0, 1.0, 1.0]
    for qt in range(3):
        sel = [(z, it["target"]) for it, z in calib if it["qtype"] == qt]
        if sel:
            try:
                temps[qt] = round(fit_one_temp(sel), 4)
            except Exception as e:  # keep training result even if calibration fails
                log("temperature fit failed for type %d: %s" % (qt, e))
    log("Temperatures (choice, score, noul): %s" % temps)

    st.set(phase="saving", message="Saving model")
    out_dir = os.path.join(run_dir, "model")
    os.makedirs(out_dir, exist_ok=True)
    sd = {k: (v.half() if v.is_floating_point() else v).contiguous().cpu() for k, v in model.state_dict().items()}
    save_file(sd, os.path.join(out_dir, "model.safetensors"))
    model.encoder.config.save_pretrained(os.path.join(out_dir, "encoder"))
    tok.save_pretrained(os.path.join(out_dir, "tokenizer"))
    _fix_tokenizer_config(out_dir)
    new_cfg = dict(cfg)
    new_cfg.update(fine_tuned=True, model_name=job["name"], base_model=base, temperature=temps,
                   max_len=max_len, head_max_len=head_max_len)
    new_cfg.pop("temperature_by_options", None)   # base buckets would override the new temperatures
    with open(os.path.join(out_dir, "rl_agent_config.json"), "w") as f:
        json.dump(new_cfg, f, indent=2)

    # schema = every question seen in the data, so the Test tab can reuse it
    schema = {}
    for rec in records:
        schema.update(rec["questions"])
    with open(os.path.join(out_dir, "questions.json"), "w") as f:
        json.dump(schema, f, indent=2, ensure_ascii=False)

    smoke = None
    try:
        st.set(message="Loading the saved model to check it works")
        del model
        if use_cuda:
            torch.cuda.empty_cache()
        import laya
        agent = laya.Agent(out_dir, device=str(device))
        rec = records[order[0]]
        smoke = agent.predict(rec["state"], rec["questions"])["answers"]
    except Exception as e:
        smoke = {"error": str(e)}
        log("Smoke test failed: %s" % e)

    metrics = {
        "name": job["name"], "base": base, "created": time.time(),
        "duration_s": int(time.time() - t_start), "gpus": world_size,
        "records": len(records), "train_records": len(records) - n_val, "val_records": n_val,
        "train_decisions": len(train_items), "val_decisions": len(val_items), "skipped": sk1 + sk2,
        "hyperparams": {"epochs": EPOCHS, "micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM,
                        "lr_encoder": LR_ENCODER, "lr_head": LR_HEAD, "max_len": max_len},
        "baseline": baseline, "final": final, "history": hist, "temperatures": temps, "smoke_test": smoke,
    }
    with open(os.path.join(run_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    st.set(phase="done", message="Finished in %d min" % (metrics["duration_s"] // 60), metrics=metrics, eta_s=0)
    log("Done. Model saved to %s" % out_dir)


if __name__ == "__main__":
    main()
''')
print("wrote", os.path.join(APP_DIR, 'train_laya.py'))

In [ ]:
# Web server for the UI
# Writes laya_studio_server.py into APP_DIR. Safe to re-run.
import os
APP_DIR = "/kaggle/working/laya_app"
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'laya_studio_server.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Laya Studio: dataset, labelling, training and testing UI for Laya on port 7860.

Standard library HTTP server (no Gradio). Start it from the notebook with
    import laya_studio_server as S; S.start(work_dir=..., port=7860, password=..., hf_token=...)
"""
import base64
import hmac
import json
import os
import re
import shutil
import signal
import socket
import subprocess
import sys
import threading
import time
import traceback
import zipfile
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, unquote, urlparse

import laya_data as D

HERE = os.path.dirname(os.path.abspath(__file__))
BASE_MODELS = [
    {"id": "convaiinnovations/laya", "label": "laya (English, 421M)"},
    {"id": "convaiinnovations/laya-typed-decisions", "label": "laya-typed-decisions (English, 421M, pre-tuned)"},
    {"id": "convaiinnovations/laya-multilingual", "label": "laya-multilingual (100+ languages incl. Nepali, 322M)"},
]
IMPORT_ROOTS = ["/kaggle/input"]
MAX_BODY = 200 * 1024 * 1024

CFG = {"work": None, "password": None, "hf_token": None}
LOCK = threading.RLock()
DATA = {"records": [], "templates": {}}
TRAIN = {"proc": None, "run": None}
AGENTS = {"key": None, "agent": None, "loading": None, "error": None}
PUSH = {}          # run name -> {"state": ..., "message": ...}
SERVERS = []


# ================================================================ storage
def p(*parts):
    return os.path.join(CFG["work"], *parts)


def _load_state():
    os.makedirs(p("runs"), exist_ok=True)
    if os.path.exists(p("dataset.jsonl")):
        recs, errs = D.parse_jsonl(open(p("dataset.jsonl"), encoding="utf-8").read())
        DATA["records"] = recs
        if errs:
            print("dataset.jsonl: skipped %d bad lines" % len(errs))
    try:
        DATA["templates"] = D.validate_templates(json.load(open(p("templates.json"))))
    except Exception:
        DATA["templates"] = D.default_templates()


def _save_records():
    tmp = p("dataset.jsonl.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        for r in DATA["records"]:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, p("dataset.jsonl"))


def _safe_name(s):
    s = re.sub(r"[^A-Za-z0-9_.-]+", "-", str(s or "")).strip("-.")
    return s[:60]


def _run_dir(name):
    name = _safe_name(name)
    if not name or not os.path.isdir(p("runs", name)):
        raise ValueError("unknown run %r" % name)
    return p("runs", name)


def _read_json(path, default=None):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return default


# ================================================================ GPU / training
def gpu_info():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5).stdout
        gpus = []
        for line in out.strip().splitlines():
            n, u, t, g = [x.strip() for x in line.split(",")]
            gpus.append({"name": n, "mem_used": int(u), "mem_total": int(t), "util": int(g)})
        return gpus
    except Exception:
        return []


def training_running():
    proc = TRAIN["proc"]
    return proc is not None and proc.poll() is None


def train_status():
    run = TRAIN["run"]
    if not run:
        return {"state": "idle"}
    rd = p("runs", run)
    st = _read_json(os.path.join(rd, "status.json"), {}) or {}
    running = training_running()
    phase = st.get("phase", "starting")
    if not running and phase not in ("done", "error"):
        phase = "stopped" if TRAIN.get("stopped") else "error"
        if phase == "error" and not st.get("error"):
            st["error"] = "Training process exited early - see the log below."
    log = ""
    try:
        with open(os.path.join(rd, "train.log"), "rb") as f:
            f.seek(0, 2)
            size = f.tell()
            f.seek(max(0, size - 12000))
            log = f.read().decode("utf-8", "replace")
    except Exception:
        pass
    st.update(run=run, state="running" if running else phase, phase=phase, log=log)
    return st


def start_training(opts):
    with LOCK:
        if training_running():
            raise ValueError("a training run is already in progress")
        if len(DATA["records"]) < 10:
            raise ValueError("add at least 10 labelled records first (you have %d)" % len(DATA["records"]))
        name = _safe_name(opts.get("name")) or time.strftime("run-%Y%m%d-%H%M%S")
        rd = p("runs", name)
        if os.path.exists(rd):
            raise ValueError("a run named %r already exists" % name)
        base = str(opts.get("base") or BASE_MODELS[0]["id"])
        if base.startswith("run:"):
            base = os.path.join(_run_dir(base[4:]), "model")
            if not os.path.exists(os.path.join(base, "model.safetensors")):
                raise ValueError("that run has no saved model")
        elif base not in [b["id"] for b in BASE_MODELS]:
            raise ValueError("unknown base model")
        os.makedirs(rd)
        shutil.copy(p("dataset.jsonl"), os.path.join(rd, "dataset.jsonl"))   # snapshot of the data used

        def num(key, default, cast, lo, hi):
            v = opts.get(key)
            v = default if v in (None, "") else cast(v)
            if not lo <= v <= hi:
                raise ValueError("%s must be between %s and %s" % (key, lo, hi))
            return v

        job = {"name": name, "run_dir": rd, "dataset": os.path.join(rd, "dataset.jsonl"), "base": base,
               "epochs": num("epochs", 4, int, 1, 50), "micro_batch": num("micro_batch", 8, int, 1, 64),
               "grad_accum": num("grad_accum", 4, int, 1, 64), "lr_encoder": num("lr_encoder", 2.5e-5, float, 1e-7, 1e-3),
               "lr_head": num("lr_head", 1e-4, float, 1e-7, 1e-2), "val_frac": num("val_frac", 0.15, float, 0.0, 0.5),
               "max_len": num("max_len", 0, int, 0, 8192), "seed": 42}
        with open(os.path.join(rd, "job.json"), "w") as f:
            json.dump(job, f, indent=2)

        _unload_agent()                       # free GPU memory for training
        n_gpu = len(gpu_info())
        want = num("gpus", n_gpu, int, 0, 16)
        n = min(n_gpu, want) if n_gpu else 0
        script = os.path.join(HERE, "train_laya.py")
        if n >= 1:
            cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node=%d" % n,
                   script, os.path.join(rd, "job.json")]
        else:
            cmd = [sys.executable, script, os.path.join(rd, "job.json")]
        env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", PYTHONUNBUFFERED="1",
                   TOKENIZERS_PARALLELISM="false")
        env.pop("OLLAMA_HOST", None)
        logf = open(os.path.join(rd, "train.log"), "w")
        logf.write("$ %s\n" % " ".join(cmd))
        logf.flush()
        TRAIN.update(proc=subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, cwd=HERE, env=env,
                                           start_new_session=True), run=name, stopped=False)
        return {"run": name, "gpus": n}


def stop_training():
    proc = TRAIN["proc"]
    if not training_running():
        return {"stopped": False}
    TRAIN["stopped"] = True
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(timeout=20)
    except Exception:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except Exception:
            pass
    return {"stopped": True}


def list_runs():
    runs = []
    for name in sorted(os.listdir(p("runs")), reverse=True):
        rd = p("runs", name)
        if not os.path.isdir(rd):
            continue
        job = _read_json(os.path.join(rd, "job.json"), {}) or {}
        st = _read_json(os.path.join(rd, "status.json"), {}) or {}
        m = _read_json(os.path.join(rd, "metrics.json"))
        has_model = os.path.exists(os.path.join(rd, "model", "model.safetensors"))
        phase = st.get("phase", "?")
        if TRAIN["run"] == name and training_running():
            phase = "running"
        elif phase not in ("done", "error"):
            phase = "stopped"
        runs.append({"name": name, "base": job.get("base"), "phase": phase, "has_model": has_model,
                     "created": os.path.getmtime(rd), "metrics": m, "push": PUSH.get(name)})
    return runs


# ================================================================ inference
def _unload_agent():
    AGENTS.update(key=None, agent=None, loading=None, error=None)
    try:
        import gc
        import torch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def _load_agent(path, device):         # replaced in tests
    import laya
    return laya.Agent(path, device=device)


def _resolve_model(key):
    if key.startswith("run:"):
        path = os.path.join(_run_dir(key[4:]), "model")
        if not os.path.exists(os.path.join(path, "model.safetensors")):
            raise ValueError("that run has no saved model yet")
        return path
    if key not in [b["id"] for b in BASE_MODELS]:
        raise ValueError("unknown model")
    return key


def get_agent(key):
    """Return a loaded agent, or None while it loads in the background (loads can exceed
    Cloudflare's 100 s request limit, so they never block a request)."""
    try:
        import torch
        device = "cuda" if torch.cuda.is_available() and not training_running() else "cpu"
    except Exception:
        device = "cpu"
    full = "%s@%s" % (key, device)
    with LOCK:
        if AGENTS["key"] == full:
            if AGENTS["agent"] is not None:
                return AGENTS["agent"]
            if AGENTS["error"]:
                err = AGENTS["error"]
                AGENTS.update(key=None, error=None)
                raise ValueError("model failed to load: " + err)
            return None
        path = _resolve_model(key)
        _unload_agent()
        AGENTS.update(key=full, loading=time.time())

    def load():
        try:
            a = _load_agent(path, device)
            with LOCK:
                if AGENTS["key"] == full:
                    AGENTS.update(agent=a, loading=None)
        except Exception as e:
            traceback.print_exc()
            with LOCK:
                if AGENTS["key"] == full:
                    AGENTS.update(error=str(e), loading=None)

    threading.Thread(target=load, daemon=True).start()
    return None


def predict(body):
    key = str(body.get("model") or BASE_MODELS[0]["id"])
    text = body.get("state")
    if isinstance(text, str):
        s = text.strip()
        state = {"text": text}
        if s[:1] == "{":
            try:
                state = json.loads(s)       # allow pasting a JSON document
            except ValueError:
                pass
    else:
        state = text
    if not state:
        raise ValueError("enter a document first")
    qs = body.get("questions") or {}
    questions = {qid: D.normalize_question(qid, q) for qid, q in qs.items()}
    if not questions:
        raise ValueError("no questions")
    agent = get_agent(key)
    if agent is None:
        return {"loading": True, "device_note": "cpu" if training_running() else None}
    t0 = time.perf_counter()
    res = agent.predict(state, questions)
    return {"loading": False, "ms": round((time.perf_counter() - t0) * 1000, 1), "result": res,
            "questions": questions}


# ================================================================ HF push
def push_run(name, repo_id, private=True):
    if not CFG["hf_token"]:
        raise ValueError("add an HF_TOKEN Kaggle secret (write access) and restart the server cell")
    rd = _run_dir(name)
    if not os.path.exists(os.path.join(rd, "model", "model.safetensors")):
        raise ValueError("run has no saved model")
    if not re.match(r"^[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+$", repo_id or ""):
        raise ValueError("repo id must look like username/model-name")
    PUSH[name] = {"state": "uploading", "message": "Uploading to %s..." % repo_id}

    def run():
        try:
            from huggingface_hub import HfApi
            api = HfApi(token=CFG["hf_token"])
            api.create_repo(repo_id, private=private, exist_ok=True)
            api.upload_folder(folder_path=os.path.join(rd, "model"), repo_id=repo_id,
                              commit_message="Laya Studio run %s" % name)
            for extra in ("metrics.json",):
                if os.path.exists(os.path.join(rd, extra)):
                    api.upload_file(path_or_fileobj=os.path.join(rd, extra), path_in_repo=extra, repo_id=repo_id)
            PUSH[name] = {"state": "done", "message": "Pushed to https://huggingface.co/%s" % repo_id}
        except Exception as e:
            PUSH[name] = {"state": "error", "message": str(e)}

    threading.Thread(target=run, daemon=True).start()
    return PUSH[name]


# ================================================================ dataset import
def import_records(recs, mode):
    with LOCK:
        if mode == "replace":
            DATA["records"] = list(recs)
        else:
            DATA["records"].extend(recs)
        _save_records()
        return len(DATA["records"])


def parse_upload(text, fmt, template):
    if fmt == "csv":
        tpl = DATA["templates"].get(template)
        if not tpl:
            raise ValueError("pick a template for CSV import")
        return D.parse_csv(text, tpl["questions"])
    return D.parse_jsonl(text)


def list_sources():
    out = []
    for root in IMPORT_ROOTS:
        for dp, dn, fn in os.walk(root):
            for f in fn:
                if f.lower().endswith((".jsonl", ".csv")):
                    fp = os.path.join(dp, f)
                    out.append({"path": fp, "size": os.path.getsize(fp)})
                    if len(out) >= 200:
                        return out
    return out


# ================================================================ HTTP
class ChunkWriter:
    """File-like object that sends HTTP chunked-encoding frames (zipfile streams into it)."""

    def __init__(self, wfile):
        self.w = wfile

    def write(self, data):
        if data:
            self.w.write(b"%x\r\n" % len(data) + bytes(data) + b"\r\n")
        return len(data)

    def flush(self):
        self.w.flush()

    def close(self):
        self.w.write(b"0\r\n\r\n")
        self.w.flush()


class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    server_version = "LayaStudio"

    def log_message(self, fmt, *args):
        pass

    # ---------------------------------------------------------- helpers
    def _authed(self):
        pw = CFG["password"]
        if not pw:
            return True
        h = self.headers.get("Authorization", "")
        if h.startswith("Basic "):
            try:
                user, _, given = base64.b64decode(h[6:]).decode("utf-8").partition(":")
                if hmac.compare_digest(given.encode(), pw.encode()):
                    return True
            except Exception:
                pass
        self.send_response(401)
        self.send_header("WWW-Authenticate", 'Basic realm="Laya Studio", charset="UTF-8"')
        self.send_header("Content-Length", "0")
        self.end_headers()
        return False

    def _send(self, code, body, ctype="application/json"):
        if not isinstance(body, (bytes, str)):
            body = json.dumps(body, ensure_ascii=False, default=str)
        data = body.encode("utf-8") if isinstance(body, str) else body
        self.send_response(code)
        self.send_header("Content-Type", ctype + ("; charset=utf-8" if ctype.startswith(("text", "application/json")) else ""))
        self.send_header("Content-Length", str(len(data)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(data)

    def _body(self):
        n = int(self.headers.get("Content-Length") or 0)
        if n > MAX_BODY:
            raise ValueError("upload too large")
        return self.rfile.read(n) if n else b""

    def _json(self):
        raw = self._body()
        return json.loads(raw.decode("utf-8")) if raw else {}

    def _stream_file(self, path, name, ctype):
        size = os.path.getsize(path)
        self.send_response(200)
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(size))
        self.send_header("Content-Disposition", 'attachment; filename="%s"' % name)
        self.end_headers()
        with open(path, "rb") as f:
            shutil.copyfileobj(f, self.wfile, 1024 * 1024)

    def _stream_zip(self, folder, name):
        self.send_response(200)
        self.send_header("Content-Type", "application/zip")
        self.send_header("Transfer-Encoding", "chunked")
        self.send_header("Content-Disposition", 'attachment; filename="%s"' % name)
        self.end_headers()
        cw = ChunkWriter(self.wfile)
        with zipfile.ZipFile(cw, "w", zipfile.ZIP_STORED, allowZip64=True) as z:
            for dp, dn, fn in os.walk(folder):
                for f in sorted(fn):
                    if f.endswith(".tmp"):
                        continue
                    fp = os.path.join(dp, f)
                    z.write(fp, os.path.relpath(fp, os.path.dirname(folder)))
        cw.close()

    # ---------------------------------------------------------- routes
    def do_GET(self):
        if not self._authed():
            return
        u = urlparse(self.path)
        qs = {k: v[0] for k, v in parse_qs(u.query).items()}
        try:
            if u.path in ("/", "/index.html"):
                with open(os.path.join(HERE, "laya_studio_ui.html"), "rb") as f:
                    return self._send(200, f.read(), "text/html")
            if u.path == "/health":
                return self._send(200, {"ok": True})
            if u.path == "/api/info":
                with LOCK:
                    stats = D.dataset_stats(DATA["records"])
                    templates = DATA["templates"]
                return self._send(200, {
                    "stats": stats, "templates": templates, "base_models": BASE_MODELS, "gpus": gpu_info(),
                    "hf": bool(CFG["hf_token"]), "train": train_status(), "work": CFG["work"],
                    "csv_examples": {t: D.csv_example(v["questions"]) for t, v in templates.items()}})
            if u.path == "/api/records":
                off, lim = int(qs.get("offset", 0)), min(200, int(qs.get("limit", 50)))
                flt = qs.get("q", "").lower()
                with LOCK:
                    rows = [(i, r) for i, r in enumerate(DATA["records"])]
                if flt:
                    rows = [(i, r) for i, r in rows
                            if flt in json.dumps([r["state"], r["gold"]], ensure_ascii=False).lower()]
                out = []
                for i, r in rows[off: off + lim]:
                    labels = {}
                    for qid, g in r["gold"].items():
                        try:
                            labels[qid] = D.gold_display(r["questions"][qid], g)
                        except ValueError:
                            labels[qid] = "?"
                    out.append({"index": i, "preview": D.state_preview(r["state"]), "labels": labels})
                return self._send(200, {"total": len(rows), "rows": out})
            if u.path == "/api/record":
                with LOCK:
                    return self._send(200, DATA["records"][int(qs["index"])])
            if u.path == "/api/dataset/download":
                with LOCK:
                    _save_records()
                return self._stream_file(p("dataset.jsonl"), "laya_dataset.jsonl", "application/x-ndjson")
            if u.path == "/api/dataset/sources":
                return self._send(200, {"files": list_sources()})
            if u.path == "/api/train/status":
                return self._send(200, train_status())
            if u.path == "/api/runs":
                return self._send(200, {"runs": list_runs()})
            if u.path.startswith("/api/runs/") and u.path.endswith("/download"):
                name = unquote(u.path.split("/")[3])
                rd = _run_dir(name)
                return self._stream_zip(rd, "laya-%s.zip" % _safe_name(name))
            if u.path.startswith("/api/runs/") and u.path.endswith("/questions"):
                rd = _run_dir(unquote(u.path.split("/")[3]))
                return self._send(200, _read_json(os.path.join(rd, "model", "questions.json"), {}))
            return self._send(404, {"error": "not found"})
        except (BrokenPipeError, ConnectionResetError):
            return
        except (ValueError, KeyError, IndexError) as e:
            return self._send(400, {"error": str(e)})
        except Exception as e:
            traceback.print_exc()
            return self._send(500, {"error": str(e)})

    def do_POST(self):
        if not self._authed():
            return
        u = urlparse(self.path)
        qs = {k: v[0] for k, v in parse_qs(u.query).items()}
        try:
            if u.path == "/api/dataset/upload":
                text = self._body().decode("utf-8-sig", "replace")
                recs, errs = parse_upload(text, qs.get("format", "jsonl"), qs.get("template"))
                total = import_records(recs, qs.get("mode", "append")) if recs else len(DATA["records"])
                return self._send(200, {"added": len(recs), "total": total, "errors": errs[:50],
                                        "error_count": len(errs)})
            if u.path == "/api/dataset/import":
                b = self._json()
                path = os.path.realpath(b.get("path", ""))
                if not any(path.startswith(os.path.realpath(r) + os.sep) for r in IMPORT_ROOTS):
                    raise ValueError("can only import files from /kaggle/input")
                text = open(path, encoding="utf-8-sig", errors="replace").read()
                fmt = "csv" if path.lower().endswith(".csv") else "jsonl"
                recs, errs = parse_upload(text, fmt, b.get("template"))
                total = import_records(recs, b.get("mode", "append")) if recs else len(DATA["records"])
                return self._send(200, {"added": len(recs), "total": total, "errors": errs[:50],
                                        "error_count": len(errs)})
            if u.path == "/api/records/add":
                b = self._json()
                rec = D.normalize_record(b)
                return self._send(200, {"total": import_records([rec], "append")})
            if u.path == "/api/records/delete":
                b = self._json()
                idx = sorted({int(i) for i in b.get("indices", [])}, reverse=True)
                with LOCK:
                    for i in idx:
                        if 0 <= i < len(DATA["records"]):
                            del DATA["records"][i]
                    _save_records()
                    return self._send(200, {"total": len(DATA["records"])})
            if u.path == "/api/records/clear":
                with LOCK:
                    DATA["records"] = []
                    _save_records()
                return self._send(200, {"total": 0})
            if u.path == "/api/templates":
                t = D.validate_templates(self._json())
                with LOCK:
                    DATA["templates"] = t
                    with open(p("templates.json"), "w") as f:
                        json.dump(t, f, indent=2, ensure_ascii=False)
                return self._send(200, {"templates": t})
            if u.path == "/api/templates/reset":
                with LOCK:
                    DATA["templates"] = D.default_templates()
                    if os.path.exists(p("templates.json")):
                        os.remove(p("templates.json"))
                return self._send(200, {"templates": DATA["templates"]})
            if u.path == "/api/train/start":
                return self._send(200, start_training(self._json()))
            if u.path == "/api/train/stop":
                return self._send(200, stop_training())
            if u.path == "/api/predict":
                return self._send(200, predict(self._json()))
            if u.path.startswith("/api/runs/") and u.path.endswith("/push"):
                b = self._json()
                return self._send(200, push_run(unquote(u.path.split("/")[3]), b.get("repo_id"), bool(b.get("private", True))))
            if u.path.startswith("/api/runs/") and u.path.endswith("/delete"):
                name = unquote(u.path.split("/")[3])
                if TRAIN["run"] == name and training_running():
                    raise ValueError("stop the run first")
                rd = _run_dir(name)
                if AGENTS["key"] and AGENTS["key"].startswith("run:%s@" % name):
                    _unload_agent()
                shutil.rmtree(rd)
                return self._send(200, {"deleted": name})
            return self._send(404, {"error": "not found"})
        except (BrokenPipeError, ConnectionResetError):
            return
        except (ValueError, KeyError, IndexError, TypeError) as e:
            return self._send(400, {"error": str(e)})
        except Exception as e:
            traceback.print_exc()
            return self._send(500, {"error": str(e)})


class V4Server(ThreadingHTTPServer):
    allow_reuse_address = True
    daemon_threads = True


class V6Server(V4Server):
    address_family = socket.AF_INET6

    def server_bind(self):
        self.socket.setsockopt(socket.IPPROTO_IPV6, socket.IPV6_V6ONLY, 1)
        super().server_bind()


def start(work_dir, port=7860, password=None, hf_token=None):
    """Start (or restart) the UI on IPv4 and IPv6 so cloudflared's 'localhost' always connects."""
    stop()
    CFG.update(work=os.path.abspath(work_dir), password=password or None, hf_token=hf_token or None)
    os.makedirs(CFG["work"], exist_ok=True)
    _load_state()
    for cls, addr in ((V4Server, "0.0.0.0"), (V6Server, "::")):
        try:
            srv = cls((addr, port), Handler)
            threading.Thread(target=srv.serve_forever, daemon=True).start()
            SERVERS.append(srv)
        except OSError as e:
            print("Could not listen on [%s]:%d -> %s" % (addr, port, e))
    if not SERVERS:
        raise RuntimeError("nothing is listening on port %d" % port)
    return SERVERS


def stop():
    while SERVERS:
        srv = SERVERS.pop()
        try:
            srv.shutdown()
            srv.server_close()
        except Exception:
            pass
''')
print("wrote", os.path.join(APP_DIR, 'laya_studio_server.py'))

In [ ]:
# UI
# Writes laya_studio_ui.html into APP_DIR. Safe to re-run.
import os
APP_DIR = "/kaggle/working/laya_app"
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'laya_studio_ui.html'), "w", encoding="utf-8") as f:
    f.write(r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Laya Studio</title>
<style>
:root{--bg:#f5f6f8;--panel:#fff;--text:#1b1f27;--muted:#6a7280;--line:#e3e6eb;--accent:#0f766e;--accent-2:#ccfbf1;--bad:#b42318;--bad-bg:#fee4e2;--ok:#067647;--ok-bg:#dcfae6;--warn:#b54708;--warn-bg:#fef0c7;--chip:#eef1f5;--mono:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace}
@media (prefers-color-scheme:dark){:root{--bg:#0e1116;--panel:#161a21;--text:#e6e9ee;--muted:#98a1ad;--line:#262c36;--accent:#2dd4bf;--accent-2:#123a36;--bad:#f97066;--bad-bg:#3b1715;--ok:#47cd89;--ok-bg:#10301f;--warn:#fdb022;--warn-bg:#3a2a0b;--chip:#222833}}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--text);font:14px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
header{background:var(--panel);border-bottom:1px solid var(--line);position:sticky;top:0;z-index:5}
.bar{max-width:1180px;margin:0 auto;padding:10px 16px;display:flex;gap:14px;align-items:center;flex-wrap:wrap}
.brand{font-weight:700;font-size:16px;letter-spacing:.2px}.brand span{color:var(--accent)}
nav{display:flex;gap:4px;flex-wrap:wrap}
nav button{background:none;border:0;padding:7px 12px;border-radius:8px;color:var(--muted);font:inherit;font-weight:600;cursor:pointer}
nav button.on{background:var(--accent-2);color:var(--accent)}
#gpu{margin-left:auto;color:var(--muted);font-size:12px}
main{max-width:1180px;margin:0 auto;padding:16px}
section{display:none}section.on{display:block}
.card{background:var(--panel);border:1px solid var(--line);border-radius:12px;padding:16px;margin-bottom:14px}
.card h2{margin:0 0 4px;font-size:15px}.card .sub{color:var(--muted);margin:0 0 12px;font-size:13px}
.grid{display:grid;gap:14px;grid-template-columns:repeat(auto-fit,minmax(320px,1fr))}
.row{display:flex;gap:8px;align-items:center;flex-wrap:wrap}
label.f{display:flex;flex-direction:column;gap:4px;font-size:12px;color:var(--muted);font-weight:600}
input,select,textarea{font:inherit;color:var(--text);background:var(--bg);border:1px solid var(--line);border-radius:8px;padding:7px 9px;min-width:0}
textarea{width:100%;resize:vertical}
select{max-width:100%;text-overflow:ellipsis}.q select{width:100%}
textarea.code{font-family:var(--mono);font-size:12px}
button.b{font:inherit;font-weight:600;border:1px solid var(--line);background:var(--panel);color:var(--text);border-radius:8px;padding:7px 12px;cursor:pointer;white-space:nowrap}
button.b:hover{border-color:var(--accent)}
button.p{background:var(--accent);border-color:var(--accent);color:#fff}
@media (prefers-color-scheme:dark){button.p{color:#062b27}}
button.d{color:var(--bad)}
button:disabled{opacity:.5;cursor:default}
a.b{display:inline-block;text-decoration:none;font-weight:600;border:1px solid var(--line);color:var(--text);border-radius:8px;padding:7px 12px}
.kpis{display:flex;gap:10px;flex-wrap:wrap;margin-bottom:10px}
.kpi{background:var(--bg);border:1px solid var(--line);border-radius:10px;padding:8px 14px;min-width:120px}
.kpi b{display:block;font-size:20px}.kpi span{color:var(--muted);font-size:12px}
.qstat{margin:10px 0}.qstat .t{font-weight:600;font-size:13px;display:flex;gap:6px;align-items:center}
.hb{display:grid;grid-template-columns:150px 1fr 48px;gap:8px;align-items:center;font-size:12px;margin:3px 0}
.hb .n{overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.track{background:var(--chip);border-radius:4px;height:10px;overflow:hidden}.fill{background:var(--accent);height:100%}
.fill.low{background:var(--warn)}
table{width:100%;border-collapse:collapse;font-size:13px}
th,td{text-align:left;padding:7px 8px;border-bottom:1px solid var(--line);vertical-align:top}
th{color:var(--muted);font-weight:600;font-size:12px}
.chip{display:inline-block;background:var(--chip);border-radius:6px;padding:1px 7px;margin:1px 3px 1px 0;font-size:12px}
.chip i{font-style:normal;color:var(--muted)}
.badge{display:inline-block;border-radius:999px;padding:2px 9px;font-size:12px;font-weight:600;background:var(--chip)}
.badge.ok{background:var(--ok-bg);color:var(--ok)}.badge.bad{background:var(--bad-bg);color:var(--bad)}.badge.run{background:var(--accent-2);color:var(--accent)}.badge.warn{background:var(--warn-bg);color:var(--warn)}
.muted{color:var(--muted)}.small{font-size:12px}
pre{background:var(--bg);border:1px solid var(--line);border-radius:8px;padding:10px;overflow:auto;font:12px/1.45 var(--mono);max-height:340px;white-space:pre-wrap;word-break:break-word;margin:8px 0 0}
.q{border:1px solid var(--line);border-radius:10px;padding:10px 12px;margin:8px 0}
.q .h{display:flex;justify-content:space-between;gap:8px;align-items:baseline}
.q .h b{font-size:13px}.q .ins{color:var(--muted);font-size:12px;margin:2px 0 8px}
.seg{display:inline-flex;border:1px solid var(--line);border-radius:8px;overflow:hidden}
.seg label{padding:5px 12px;cursor:pointer;font-size:13px;border-right:1px solid var(--line)}
.seg label:last-child{border-right:0}
.seg input{display:none}.seg input:checked+span{color:var(--accent);font-weight:700}
.prog{height:8px;background:var(--chip);border-radius:6px;overflow:hidden;margin:8px 0}
.prog div{height:100%;background:var(--accent);transition:width .4s}
.ans{display:flex;gap:10px;align-items:baseline;flex-wrap:wrap}
.ans .v{font-size:17px;font-weight:700;color:var(--accent)}
#toast{position:fixed;bottom:18px;left:50%;transform:translateX(-50%);background:var(--text);color:var(--bg);padding:9px 16px;border-radius:10px;font-weight:600;opacity:0;transition:opacity .2s;pointer-events:none;z-index:20;max-width:90vw}
#toast.show{opacity:1}#toast.bad{background:var(--bad);color:#fff}
dialog{border:1px solid var(--line);border-radius:12px;background:var(--panel);color:var(--text);max-width:min(860px,94vw);width:100%}
dialog::backdrop{background:rgba(0,0,0,.4)}
details summary{cursor:pointer;font-weight:600}
.empty{color:var(--muted);text-align:center;padding:24px}
@media (max-width:640px){.hb{grid-template-columns:100px 1fr 40px}main{padding:10px}}
</style>
</head>
<body>
<header><div class="bar">
  <div class="brand">Laya <span>Studio</span></div>
  <nav id="nav">
    <button data-t="dataset" class="on">Dataset</button>
    <button data-t="label">Label</button>
    <button data-t="train">Train</button>
    <button data-t="test">Test</button>
    <button data-t="models">Models</button>
  </nav>
  <div id="gpu"></div>
</div></header>

<main>
<!-- ============================================================ DATASET -->
<section id="t-dataset" class="on">
  <div class="card">
    <h2>Your dataset</h2>
    <p class="sub">Each record is one document (bill, receipt, transaction, message) with the answers you labelled for it.</p>
    <div class="kpis" id="kpis"></div>
    <div id="qstats"></div>
  </div>

  <div class="grid">
    <div class="card">
      <h2>Upload</h2>
      <p class="sub">JSONL in Laya format, or a CSV with a <code>text</code> column plus one column per question.</p>
      <div class="row">
        <input type="file" id="upFile" accept=".jsonl,.json,.csv,.txt">
      </div>
      <div class="row" style="margin-top:10px">
        <label class="f">CSV template<select id="csvTpl"></select></label>
        <label class="f">Mode<select id="upMode"><option value="append">Append</option><option value="replace">Replace all</option></select></label>
        <button class="b p" id="upBtn" style="align-self:flex-end">Upload</button>
      </div>
      <div id="upOut" class="small" style="margin-top:8px"></div>
      <details style="margin-top:10px"><summary class="small">File formats</summary>
        <p class="small muted">JSONL, one record per line:</p>
        <pre id="jsonlEx"></pre>
        <p class="small muted">CSV for the selected template (labels may be the option name, yes/no, or a level number):</p>
        <pre id="csvEx"></pre>
        <p class="small muted">Extra CSV columns (merchant, amount, date...) are passed to the model with the text. Cloudflare limits uploads to 100 MB; for bigger files attach them to the notebook as a Kaggle dataset and use the import list.</p>
      </details>
    </div>

    <div class="card">
      <h2>Import from Kaggle input</h2>
      <p class="sub">Files attached to this notebook under /kaggle/input. Keeps your data safe between sessions.</p>
      <button class="b" id="scanBtn">Scan /kaggle/input</button>
      <div id="sources" class="small" style="margin-top:8px"></div>
    </div>
  </div>

  <div class="card">
    <div class="row" style="justify-content:space-between">
      <h2 style="margin:0">Records</h2>
      <div class="row">
        <input id="recSearch" placeholder="Search..." style="width:180px">
        <a class="b" href="/api/dataset/download">Download JSONL</a>
        <button class="b d" id="clearBtn">Clear all</button>
      </div>
    </div>
    <div style="overflow-x:auto;margin-top:10px"><table>
      <thead><tr><th>#</th><th>Document</th><th>Labels</th><th></th></tr></thead>
      <tbody id="recRows"></tbody>
    </table></div>
    <div class="row" style="justify-content:flex-end;margin-top:10px">
      <span id="recPage" class="muted small"></span>
      <button class="b" id="prevBtn">Previous</button><button class="b" id="nextBtn">Next</button>
    </div>
  </div>
</section>

<!-- ============================================================ LABEL -->
<section id="t-label">
  <div class="grid">
    <div class="card">
      <h2>Label a document</h2>
      <p class="sub">Paste the text of a bill, receipt or transaction (OCR output works), answer the questions, save. Skip any question you're unsure about.</p>
      <div class="row">
        <label class="f">Template<select id="lTpl"></select></label>
        <label class="f">Pre-fill with<select id="lModel"></select></label>
        <button class="b" id="lPrefill" style="align-self:flex-end">Pre-fill answers</button>
      </div>
      <textarea id="lText" rows="12" style="margin-top:10px" placeholder="e.g.  Bhatbhateni Supermarket, Lalitpur ... Sub total 2,070.80  VAT 13% 269.20  Grand total 2,340.00  Paid: eSewa"></textarea>
      <div class="row" style="margin-top:10px">
        <button class="b p" id="lSave">Save example</button>
        <span class="muted small">Ctrl+Enter to save</span>
        <span id="lCount" class="muted small" style="margin-left:auto"></span>
      </div>
    </div>
    <div class="card">
      <h2>Answers</h2>
      <div id="lQs"></div>
    </div>
  </div>
  <div class="card">
    <details><summary>Edit question templates</summary>
      <p class="sub" style="margin-top:8px">Types: <b>choice</b> (criteria = options with descriptions), <b>noul</b> (yes/no), <b>score</b> (criteria = ordered list of levels). Good descriptions help the model a lot.</p>
      <textarea id="tplJson" class="code" rows="18"></textarea>
      <div class="row" style="margin-top:8px"><button class="b p" id="tplSave">Save templates</button><button class="b" id="tplReset">Reset to defaults</button></div>
    </details>
  </div>
</section>

<!-- ============================================================ TRAIN -->
<section id="t-train">
  <div class="grid">
    <div class="card">
      <h2>Fine-tune</h2>
      <p class="sub">Trains on the current dataset. Part of it is held back to measure accuracy before and after.</p>
      <div class="grid" style="grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px">
        <label class="f" style="grid-column:1/-1">Start from<select id="trBase"></select></label>
        <label class="f">Run name<input id="trName" placeholder="auto"></label>
        <label class="f">Epochs<input id="trEpochs" type="number" value="4" min="1" max="50"></label>
        <label class="f">Validation %<input id="trVal" type="number" value="15" min="0" max="50"></label>
        <label class="f">Micro batch<input id="trMb" type="number" value="8" min="1" max="64"></label>
        <label class="f">Grad accumulation<input id="trGa" type="number" value="4" min="1" max="64"></label>
        <label class="f">GPUs<select id="trGpus"></select></label>
        <label class="f">LR encoder<input id="trLrE" value="2.5e-5"></label>
        <label class="f">LR head<input id="trLrH" value="1e-4"></label>
        <label class="f">Max tokens<input id="trMax" placeholder="model default"></label>
      </div>
      <div class="row" style="margin-top:12px">
        <button class="b p" id="trStart">Start training</button>
        <button class="b d" id="trStop">Stop</button>
      </div>
      <p class="small muted" style="margin-bottom:0">Rough guide: a few hundred labelled documents per question is a good start. On 2×T4 the official recipe ran ~30k decisions × 4 epochs in 4–5 h; small datasets finish in minutes.</p>
    </div>
    <div class="card">
      <div class="row" style="justify-content:space-between"><h2 style="margin:0">Progress</h2><span id="trBadge" class="badge">idle</span></div>
      <p id="trMsg" class="sub" style="margin-top:6px"></p>
      <div class="prog"><div id="trProg" style="width:0"></div></div>
      <div class="kpis" id="trKpis"></div>
      <div id="trHist"></div>
      <pre id="trErr" style="display:none;color:var(--bad)"></pre>
    </div>
  </div>
  <div class="card"><details id="logBox"><summary>Training log</summary><pre id="trLog" style="max-height:420px"></pre></details></div>
</section>

<!-- ============================================================ TEST -->
<section id="t-test">
  <div class="grid">
    <div class="card">
      <h2>Try a model</h2>
      <div class="row">
        <label class="f">Model<select id="tModel"></select></label>
        <label class="f">Questions<select id="tTpl"></select></label>
        <button class="b" id="tFromRun" style="align-self:flex-end">Use this run's questions</button>
      </div>
      <textarea id="tText" rows="9" style="margin-top:10px" placeholder="Paste a document (plain text or JSON)"></textarea>
      <details style="margin-top:8px"><summary class="small">Questions (JSON, editable)</summary><textarea id="tQs" class="code" rows="12" style="margin-top:6px"></textarea></details>
      <div class="row" style="margin-top:10px"><button class="b p" id="tRun">Run</button><span id="tInfo" class="muted small"></span></div>
    </div>
    <div class="card">
      <h2>Answers</h2>
      <div id="tOut"><div class="empty">Run a document to see the model's decisions and confidence.</div></div>
      <details style="margin-top:8px"><summary class="small">Raw JSON</summary><pre id="tRaw"></pre></details>
    </div>
  </div>
</section>

<!-- ============================================================ MODELS -->
<section id="t-models">
  <div class="card">
    <div class="row" style="justify-content:space-between"><h2 style="margin:0">Trained models</h2><button class="b" id="mRefresh">Refresh</button></div>
    <p class="sub" style="margin-top:6px">Everything in /kaggle/working disappears when the Kaggle session ends. Download the zip or push to Hugging Face to keep a model.</p>
    <div style="overflow-x:auto"><table>
      <thead><tr><th>Run</th><th>Started from</th><th>Status</th><th>Data</th><th>Val accuracy</th><th></th></tr></thead>
      <tbody id="mRows"></tbody>
    </table></div>
  </div>
</section>
</main>

<dialog id="dlg"><div class="row" style="justify-content:space-between"><b id="dlgTitle"></b><button class="b" onclick="dlg.close()">Close</button></div><pre id="dlgBody" style="max-height:70vh"></pre></dialog>
<div id="toast"></div>

<script>
const $ = s => document.querySelector(s);
const esc = s => String(s ?? '').replace(/[&<>"']/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
let INFO = null, TAB = 'dataset', recOff = 0, RUNS = [];
const PAGE = 50;

async function api(path, opts = {}) {
  const r = await fetch(path, opts);
  let d = {};
  try { d = await r.json(); } catch (e) {}
  if (!r.ok) throw new Error(d.error || (r.status + ' ' + r.statusText));
  return d;
}
const post = (path, body) => api(path, {method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify(body || {})});
let toastT;
function toast(msg, bad) {
  const t = $('#toast'); t.textContent = msg; t.className = 'show' + (bad ? ' bad' : '');
  clearTimeout(toastT); toastT = setTimeout(() => t.className = '', bad ? 6000 : 2500);
}
async function guard(fn) { try { await fn(); } catch (e) { toast(e.message, true); } }
const pct = v => v == null ? '–' : (v * 100).toFixed(1) + '%';
const fmtS = s => { s = Math.max(0, Math.round(s || 0)); const h = Math.floor(s / 3600), m = Math.floor(s % 3600 / 60), x = s % 60; return (h ? h + 'h ' : '') + m + 'm ' + String(x).padStart(2, '0') + 's'; };
const shortBase = b => !b ? '' : b.includes('/runs/') ? 'run: ' + b.split('/runs/')[1].split('/')[0] : b.replace('convaiinnovations/', '');

// ------------------------------------------------------------ tabs
$('#nav').onclick = e => {
  const b = e.target.closest('button'); if (!b) return;
  TAB = b.dataset.t;
  document.querySelectorAll('#nav button').forEach(x => x.classList.toggle('on', x === b));
  document.querySelectorAll('section').forEach(s => s.classList.toggle('on', s.id === 't-' + TAB));
  if (TAB === 'train') refreshTrain();
  if (TAB === 'models' || TAB === 'test' || TAB === 'train') refreshRuns();
  if (TAB === 'dataset') { loadInfo(); loadRecords(); }
};

// ------------------------------------------------------------ info
function fillSelect(sel, items, keep) {
  const old = sel.value;
  sel.innerHTML = items.map(([v, l]) => `<option value="${esc(v)}">${esc(l)}</option>`).join('');
  if (keep && items.some(([v]) => v === old)) sel.value = old;
}
async function loadInfo() {
  INFO = await api('/api/info');
  const tpls = Object.entries(INFO.templates).map(([k, v]) => [k, v.title]);
  ['#csvTpl', '#lTpl', '#tTpl'].forEach(s => fillSelect($(s), tpls, true));
  const g = INFO.gpus;
  $('#gpu').textContent = g.length ? g.map(x => `${x.name} ${Math.round(x.mem_used / 1024 * 10) / 10}/${Math.round(x.mem_total / 1024)} GB ${x.util}%`).join(' · ') : 'No GPU - enable a GPU accelerator to train';
  fillSelect($('#trGpus'), g.length ? g.map((_, i) => [String(i + 1), String(i + 1)]).reverse() : [['0', 'CPU (very slow)']], true);
  renderStats(INFO.stats);
  $('#lCount').textContent = INFO.stats.records + ' records';
  $('#csvEx').textContent = INFO.csv_examples[$('#csvTpl').value] || '';
  $('#jsonlEx').textContent = '{"state": {"text": "Receipt ... Total Rs 2,340 incl. VAT, paid via eSewa"},\n "questions": {"doc_type": {"type": "choice", "instructions": "What kind of document is this?",\n                            "criteria": {"invoice": "...", "receipt": "...", "other": "..."}},\n               "vat_charged": {"type": "noul", "instructions": "Is VAT charged?"}},\n "gold": {"doc_type": "receipt", "vat_charged": true}}';
  if (!$('#tplJson').dataset.dirty) $('#tplJson').value = JSON.stringify(INFO.templates, null, 2);
  if (!$('#lQs').dataset.tpl) renderLabelQs();
  if (!$('#tQs').value) setTestQs();
}
$('#csvTpl').onchange = () => { $('#csvEx').textContent = INFO.csv_examples[$('#csvTpl').value] || ''; };

function renderStats(s) {
  $('#kpis').innerHTML = `<div class="kpi"><b>${s.records}</b><span>documents</span></div><div class="kpi"><b>${s.decisions}</b><span>labelled answers</span></div><div class="kpi"><b>${Object.keys(s.questions).length}</b><span>questions</span></div>`;
  const qs = Object.entries(s.questions);
  if (!qs.length) { $('#qstats').innerHTML = '<div class="empty">No data yet. Upload a file or label documents in the Label tab.</div>'; return; }
  $('#qstats').innerHTML = qs.map(([qid, d]) => {
    const tot = Object.values(d.counts).reduce((a, b) => a + b, 0);
    const rows = Object.entries(d.counts).sort((a, b) => b[1] - a[1]).map(([lab, n]) =>
      `<div class="hb"><span class="n" title="${esc(lab)}">${esc(lab)}</span><div class="track"><div class="fill${n < 5 ? ' low' : ''}" style="width:${(n / tot * 100).toFixed(1)}%"></div></div><span>${n}</span></div>`).join('');
    const few = Object.values(d.counts).some(n => n < 5);
    return `<div class="qstat"><div class="t">${esc(qid)} <span class="badge">${esc(d.type)}</span>${few ? '<span class="badge warn">some labels have &lt;5 examples</span>' : ''}</div>${rows}</div>`;
  }).join('');
}

// ------------------------------------------------------------ records
async function loadRecords() {
  const q = encodeURIComponent($('#recSearch').value.trim());
  const d = await api(`/api/records?offset=${recOff}&limit=${PAGE}&q=${q}`);
  $('#recRows').innerHTML = d.rows.length ? d.rows.map(r => `<tr><td class="muted">${r.index + 1}</td><td>${esc(r.preview)}</td><td>${Object.entries(r.labels).map(([k, v]) => `<span class="chip"><i>${esc(k)}:</i> ${esc(v)}</span>`).join('')}</td><td style="white-space:nowrap"><button class="b" data-view="${r.index}">View</button> <button class="b d" data-del="${r.index}">Delete</button></td></tr>`).join('') : '<tr><td colspan="4" class="empty">No records</td></tr>';
  $('#recPage').textContent = d.total ? `${recOff + 1}–${Math.min(recOff + PAGE, d.total)} of ${d.total}` : '';
  $('#prevBtn').disabled = recOff === 0; $('#nextBtn').disabled = recOff + PAGE >= d.total;
}
$('#recRows').onclick = e => guard(async () => {
  const v = e.target.dataset.view, dl = e.target.dataset.del;
  if (v !== undefined) { const r = await api('/api/record?index=' + v); $('#dlgTitle').textContent = 'Record ' + (+v + 1); $('#dlgBody').textContent = JSON.stringify(r, null, 2); $('#dlg').showModal(); }
  if (dl !== undefined && confirm('Delete record ' + (+dl + 1) + '?')) { await post('/api/records/delete', {indices: [+dl]}); await loadRecords(); await loadInfo(); }
});
$('#prevBtn').onclick = () => { recOff = Math.max(0, recOff - PAGE); guard(loadRecords); };
$('#nextBtn').onclick = () => { recOff += PAGE; guard(loadRecords); };
let srchT; $('#recSearch').oninput = () => { clearTimeout(srchT); srchT = setTimeout(() => { recOff = 0; guard(loadRecords); }, 300); };
$('#clearBtn').onclick = () => guard(async () => { if (!confirm('Delete ALL records? Download a copy first if you need it.')) return; await post('/api/records/clear'); recOff = 0; await loadInfo(); await loadRecords(); toast('Dataset cleared'); });

function showImport(d) {
  $('#upOut').innerHTML = `<span class="badge ${d.added ? 'ok' : 'bad'}">${d.added} added</span> ${d.total} total` + (d.error_count ? ` · <b style="color:var(--bad)">${d.error_count} skipped</b><pre>${esc(d.errors.join('\n'))}${d.error_count > d.errors.length ? '\n…' : ''}</pre>` : '');
}
$('#upBtn').onclick = () => guard(async () => {
  const f = $('#upFile').files[0]; if (!f) throw new Error('Choose a file first');
  if ($('#upMode').value === 'replace' && !confirm('Replace the whole dataset with this file?')) return;
  const fmt = f.name.toLowerCase().endsWith('.csv') ? 'csv' : 'jsonl';
  $('#upBtn').disabled = true;
  try {
    const d = await api(`/api/dataset/upload?format=${fmt}&template=${encodeURIComponent($('#csvTpl').value)}&mode=${$('#upMode').value}`, {method: 'POST', body: await f.text()});
    showImport(d); await loadInfo(); recOff = 0; await loadRecords();
  } finally { $('#upBtn').disabled = false; }
});
$('#scanBtn').onclick = () => guard(async () => {
  const d = await api('/api/dataset/sources');
  $('#sources').innerHTML = d.files.length ? `<table>${d.files.map(f => `<tr><td style="word-break:break-all">${esc(f.path)}</td><td class="muted">${(f.size / 1048576).toFixed(1)} MB</td><td><button class="b" data-imp="${esc(f.path)}">Import</button></td></tr>`).join('')}</table><p class="muted">CSV files use the CSV template and mode from the Upload card.</p>` : '<p class="muted">No .jsonl or .csv files found. Use "Add Input" in the Kaggle sidebar to attach a dataset.</p>';
});
$('#sources').onclick = e => guard(async () => {
  const path = e.target.dataset.imp; if (!path) return;
  const d = await post('/api/dataset/import', {path, template: $('#csvTpl').value, mode: $('#upMode').value});
  showImport(d); await loadInfo(); recOff = 0; await loadRecords();
});

// ------------------------------------------------------------ label
function qControl(qid, q, prefix) {
  let c = '';
  if (q.type === 'choice') c = `<select id="${prefix}${esc(qid)}"><option value="">— skip —</option>${Object.entries(q.criteria).map(([k, v]) => `<option value="${esc(k)}" title="${esc(v)}">${esc(k)}${v ? ' — ' + esc(v) : ''}</option>`).join('')}</select>`;
  else if (q.type === 'score') c = `<select id="${prefix}${esc(qid)}"><option value="">— skip —</option>${q.criteria.map((v, i) => `<option value="${i}">${i}: ${esc(v)}</option>`).join('')}</select>`;
  else c = `<div class="seg">${[['', 'skip'], ['true', 'yes'], ['false', 'no']].map(([v, l]) => `<label><input type="radio" name="${prefix}${esc(qid)}" value="${v}"${v === '' ? ' checked' : ''}><span>${l}</span></label>`).join('')}</div>`;
  return `<div class="q"><div class="h"><b>${esc(qid)}</b><span class="badge">${esc(q.type)}</span></div><div class="ins">${esc(q.instructions)}</div>${c}<div class="small muted" id="${prefix}hint-${esc(qid)}"></div></div>`;
}
function curLabelTpl() { return INFO.templates[$('#lTpl').value] || Object.values(INFO.templates)[0]; }
function renderLabelQs() {
  const t = curLabelTpl(); $('#lQs').dataset.tpl = $('#lTpl').value;
  $('#lQs').innerHTML = Object.entries(t.questions).map(([qid, q]) => qControl(qid, q, 'lq-')).join('');
}
$('#lTpl').onchange = renderLabelQs;
function readLabel(qid, q) {
  if (q.type === 'noul') { const r = document.querySelector(`input[name="lq-${CSS.escape(qid)}"]:checked`); return r && r.value ? r.value === 'true' : null; }
  const v = document.getElementById('lq-' + qid).value; if (v === '') return null;
  return q.type === 'score' ? +v : v;
}
function stateFrom(text) {
  const s = text.trim();
  if (s.startsWith('{')) { try { return JSON.parse(s); } catch (e) {} }
  return {text: text};
}
$('#lSave').onclick = () => guard(async () => {
  const text = $('#lText').value; if (!text.trim()) throw new Error('Paste the document text first');
  const t = curLabelTpl(), questions = {}, gold = {};
  for (const [qid, q] of Object.entries(t.questions)) { const v = readLabel(qid, q); if (v !== null) { questions[qid] = q; gold[qid] = v; } }
  if (!Object.keys(gold).length) throw new Error('Answer at least one question');
  const d = await post('/api/records/add', {state: stateFrom(text), questions, gold});
  $('#lText').value = ''; renderLabelQs(); $('#lText').focus();
  $('#lCount').textContent = d.total + ' records'; toast('Saved · ' + d.total + ' records');
});
$('#lText').onkeydown = e => { if (e.key === 'Enter' && (e.ctrlKey || e.metaKey)) { e.preventDefault(); $('#lSave').click(); } };
async function runPredict(model, state, questions, onWait) {
  for (let i = 0; i < 200; i++) {
    const d = await post('/api/predict', {model, state, questions});
    if (!d.loading) return d;
    onWait && onWait(i);
    await new Promise(r => setTimeout(r, 3000));
  }
  throw new Error('Model is taking too long to load');
}
$('#lPrefill').onclick = () => guard(async () => {
  const text = $('#lText').value; if (!text.trim()) throw new Error('Paste the document text first');
  const t = curLabelTpl(); $('#lPrefill').disabled = true;
  try {
    const d = await runPredict($('#lModel').value, text, t.questions, i => $('#lPrefill').textContent = 'Loading model' + '.'.repeat(i % 4));
    for (const [qid, a] of Object.entries(d.result.answers)) {
      const q = t.questions[qid]; if (!q) continue;
      const hint = document.getElementById('lq-hint-' + qid);
      if (q.type === 'noul') { const v = a.noul >= 0.5 ? 'true' : 'false'; const r = document.querySelector(`input[name="lq-${CSS.escape(qid)}"][value="${v}"]`); if (r) r.checked = true; hint.textContent = `model: P(yes)=${a.noul}`; }
      else if (q.type === 'choice') { document.getElementById('lq-' + qid).value = a.choice; hint.textContent = `model: ${a.choice} (${pct(a.confidence)} confident)`; }
      else { const probs = a.probabilities || {}; let best = Math.round(a.score); let bp = -1; for (const [k, v] of Object.entries(probs)) if (v > bp) { bp = v; best = +k; } document.getElementById('lq-' + qid).value = String(best); hint.textContent = `model: level ${best} (expected ${a.score})`; }
    }
    toast('Pre-filled - check each answer before saving');
  } finally { $('#lPrefill').disabled = false; $('#lPrefill').textContent = 'Pre-fill answers'; }
});
$('#tplJson').oninput = () => $('#tplJson').dataset.dirty = '1';
$('#tplSave').onclick = () => guard(async () => {
  let t; try { t = JSON.parse($('#tplJson').value); } catch (e) { throw new Error('Invalid JSON: ' + e.message); }
  const d = await post('/api/templates', t); delete $('#tplJson').dataset.dirty;
  $('#tplJson').value = JSON.stringify(d.templates, null, 2); await loadInfo(); renderLabelQs(); setTestQs(); toast('Templates saved');
});
$('#tplReset').onclick = () => guard(async () => { if (!confirm('Reset templates to the built-in ones?')) return; await post('/api/templates/reset'); delete $('#tplJson').dataset.dirty; $('#tplJson').value = ''; await loadInfo(); $('#tplJson').value = JSON.stringify(INFO.templates, null, 2); renderLabelQs(); setTestQs(); });

// ------------------------------------------------------------ runs / model selects
async function refreshRuns() {
  const d = await api('/api/runs'); RUNS = d.runs;
  const bases = INFO ? INFO.base_models.map(b => [b.id, b.label]) : [];
  const runModels = RUNS.filter(r => r.has_model).map(r => ['run:' + r.name, 'Fine-tuned: ' + r.name]);
  fillSelect($('#trBase'), bases.concat(runModels.map(([v, l]) => [v, 'Continue from ' + l.slice(12)])), true);
  fillSelect($('#tModel'), runModels.concat(bases), true);
  fillSelect($('#lModel'), runModels.concat(bases), true);
  renderRuns();
}
function renderRuns() {
  $('#mRows').innerHTML = RUNS.length ? RUNS.map(r => {
    const m = r.metrics || {};
    const acc = m.final ? `${pct(m.baseline && m.baseline.accuracy)} → <b>${pct(m.final.accuracy)}</b>` : '–';
    const badge = {done: 'ok', error: 'bad', running: 'run', stopped: 'warn'}[r.phase] || '';
    const push = r.push ? `<div class="small ${r.push.state === 'error' ? '' : 'muted'}" style="${r.push.state === 'error' ? 'color:var(--bad)' : ''}">${esc(r.push.message)}</div>` : '';
    return `<tr><td><b>${esc(r.name)}</b><div class="small muted">${new Date(r.created * 1000).toLocaleString()}</div></td><td>${esc(shortBase(r.base))}</td><td><span class="badge ${badge}">${esc(r.phase)}</span></td><td class="small">${m.records ? `${m.train_records} train / ${m.val_records} val docs` : ''}</td><td>${acc}</td><td style="white-space:nowrap">${r.has_model ? `<button class="b" data-test="${esc(r.name)}">Test</button> <a class="b" href="/api/runs/${encodeURIComponent(r.name)}/download">Download</a> <button class="b" data-push="${esc(r.name)}">Push to HF</button> ` : ''}<button class="b" data-info="${esc(r.name)}">Details</button> <button class="b d" data-rm="${esc(r.name)}">Delete</button>${push}</td></tr>`;
  }).join('') : '<tr><td colspan="6" class="empty">No runs yet</td></tr>';
}
$('#mRows').onclick = e => guard(async () => {
  const ds = e.target.dataset;
  if (ds.test) { $('#tModel').value = 'run:' + ds.test; document.querySelector('#nav [data-t="test"]').click(); await loadRunQs(); }
  if (ds.info) { const r = RUNS.find(x => x.name === ds.info); $('#dlgTitle').textContent = ds.info; $('#dlgBody').textContent = JSON.stringify(r.metrics || r, null, 2); $('#dlg').showModal(); }
  if (ds.push) {
    if (!INFO.hf) throw new Error('Add an HF_TOKEN secret in Kaggle (Add-ons → Secrets) and re-run the server cell');
    const repo = prompt('Hugging Face repo id (username/model-name). It will be created as private.', 'your-username/laya-' + ds.push);
    if (!repo) return; await post(`/api/runs/${encodeURIComponent(ds.push)}/push`, {repo_id: repo, private: true}); await refreshRuns();
    const poll = setInterval(async () => { await refreshRuns(); const r = RUNS.find(x => x.name === ds.push); if (!r || !r.push || r.push.state !== 'uploading') clearInterval(poll); }, 4000);
  }
  if (ds.rm && confirm(`Delete run ${ds.rm} and its model?`)) { await post(`/api/runs/${encodeURIComponent(ds.rm)}/delete`); await refreshRuns(); }
});
$('#mRefresh').onclick = () => guard(refreshRuns);

// ------------------------------------------------------------ train
$('#trStart').onclick = () => guard(async () => {
  const body = {base: $('#trBase').value, name: $('#trName').value.trim(), epochs: $('#trEpochs').value, val_frac: (+$('#trVal').value || 0) / 100,
    micro_batch: $('#trMb').value, grad_accum: $('#trGa').value, gpus: $('#trGpus').value, lr_encoder: $('#trLrE').value, lr_head: $('#trLrH').value, max_len: $('#trMax').value};
  const d = await post('/api/train/start', body);
  toast(`Started ${d.run} on ${d.gpus || 'CPU'} ${d.gpus ? 'GPU(s)' : ''}`); $('#trName').value = '';
  await refreshTrain(); refreshRuns();
});
$('#trStop').onclick = () => guard(async () => { if (!confirm('Stop training? Progress is lost.')) return; await post('/api/train/stop'); await refreshTrain(); });
let lastState = null;
async function refreshTrain() {
  const s = await api('/api/train/status');
  const state = s.state;
  const badge = {running: 'run', done: 'ok', error: 'bad', stopped: 'warn'}[state] || '';
  $('#trBadge').className = 'badge ' + badge;
  $('#trBadge').textContent = state === 'running' ? (s.phase || 'running').replace('_', ' ') : state;
  $('#trStart').disabled = state === 'running'; $('#trStop').disabled = state !== 'running';
  $('#trMsg').textContent = s.run ? `${s.run} · ${s.message || ''}` : 'No training started yet.';
  const frac = s.total_steps ? s.step / s.total_steps : (state === 'done' ? 1 : 0);
  $('#trProg').style.width = (frac * 100).toFixed(1) + '%';
  const k = [];
  if (s.epochs) k.push([`${s.epoch || 0}/${s.epochs}`, 'epoch']);
  if (s.total_steps) k.push([`${s.step || 0}/${s.total_steps}`, 'steps']);
  if (s.loss != null) k.push([s.loss, 'loss']);
  if (state === 'running' && s.eta_s) k.push([fmtS(s.eta_s), 'remaining']);
  if (s.started) k.push([fmtS(((state === 'running' ? Date.now() / 1000 : s.updated) || s.started) - s.started), 'elapsed']);
  $('#trKpis').innerHTML = k.map(([v, l]) => `<div class="kpi"><b>${esc(v)}</b><span>${l}</span></div>`).join('');
  let h = '';
  const base = s.baseline || (s.metrics && s.metrics.baseline), hist = s.history || [];
  if (base || hist.length) {
    h += '<table><thead><tr><th>Checkpoint</th><th>Train loss</th><th>Val accuracy</th></tr></thead><tbody>';
    if (base) h += `<tr><td>Before training</td><td>–</td><td>${pct(base.accuracy)}</td></tr>`;
    h += hist.map(e => `<tr><td>Epoch ${e.epoch}</td><td>${e.train_loss}</td><td>${pct(e.val_accuracy)}</td></tr>`).join('') + '</tbody></table>';
  }
  const m = s.metrics;
  if (m && m.final) {
    h += '<p class="small" style="margin:12px 0 4px"><b>Per question (validation)</b></p><table><thead><tr><th>Question</th><th>Before</th><th>After</th><th>n</th></tr></thead><tbody>' +
      Object.entries(m.final.per_question).map(([q, v]) => `<tr><td>${esc(q)}</td><td>${pct(m.baseline && m.baseline.per_question[q] && m.baseline.per_question[q].accuracy)}</td><td><b>${pct(v.accuracy)}</b></td><td>${v.n}</td></tr>`).join('') + '</tbody></table>';
  }
  if (m && !m.final && state === 'done') h += '<p class="small muted">No validation split, so no accuracy was measured. Try the model in the Test tab.</p>';
  if (m && m.smoke_test && m.smoke_test.error) h += `<p class="small" style="color:var(--bad)">Saved model failed to load: ${esc(m.smoke_test.error)}</p>`;
  $('#trHist').innerHTML = h;
  $('#trErr').style.display = s.error ? 'block' : 'none'; $('#trErr').textContent = s.error || '';
  const lg = $('#trLog'); const atEnd = lg.scrollTop + lg.clientHeight >= lg.scrollHeight - 30;
  lg.textContent = s.log || ''; if (atEnd) lg.scrollTop = lg.scrollHeight;
  if (lastState === 'running' && state !== 'running') { refreshRuns(); toast('Training ' + state, state !== 'done'); }
  lastState = state;
  return s;
}

// ------------------------------------------------------------ test
function setTestQs() { const t = INFO && INFO.templates[$('#tTpl').value]; if (t) $('#tQs').value = JSON.stringify(t.questions, null, 2); }
$('#tTpl').onchange = setTestQs;
async function loadRunQs() {
  const v = $('#tModel').value; if (!v.startsWith('run:')) throw new Error('Pick a fine-tuned run first');
  const q = await api(`/api/runs/${encodeURIComponent(v.slice(4))}/questions`);
  if (!Object.keys(q).length) throw new Error('This run has no saved questions');
  $('#tQs').value = JSON.stringify(q, null, 2); toast('Loaded the questions this model was trained on');
}
$('#tFromRun').onclick = () => guard(loadRunQs);
function bars(entries, top) {
  return entries.map(([k, v]) => `<div class="hb"><span class="n" title="${esc(k)}">${k === top ? '<b>' + esc(k) + '</b>' : esc(k)}</span><div class="track"><div class="fill" style="width:${(v * 100).toFixed(1)}%"></div></div><span>${(v * 100).toFixed(0)}%</span></div>`).join('');
}
$('#tRun').onclick = () => guard(async () => {
  const text = $('#tText').value; if (!text.trim()) throw new Error('Paste a document first');
  let qs; try { qs = JSON.parse($('#tQs').value); } catch (e) { throw new Error('Questions JSON is invalid: ' + e.message); }
  $('#tRun').disabled = true; $('#tInfo').textContent = '';
  try {
    const d = await runPredict($('#tModel').value, text, qs, i => $('#tInfo').textContent = 'Loading model (the first load downloads ~1 GB)' + '.'.repeat(i % 4));
    $('#tInfo').textContent = `${d.ms} ms`;
    $('#tRaw').textContent = JSON.stringify(d.result, null, 2);
    $('#tOut').innerHTML = Object.entries(d.result.answers).map(([qid, a]) => {
      const q = d.questions[qid] || {};
      let head = '', body = '';
      if (q.type === 'choice') { head = a.choice; body = bars(Object.entries(a.probabilities).sort((x, y) => y[1] - x[1]), a.choice); }
      else if (q.type === 'noul') { head = a.noul >= 0.5 ? 'yes' : 'no'; body = bars([['yes', a.noul], ['no', 1 - a.noul]], head); }
      else { head = 'level ' + a.score; body = bars(Object.entries(a.probabilities).map(([k, v]) => [k + ': ' + String((q.criteria || [])[+k] || '').split(':')[0], v]), null); }
      return `<div class="q"><div class="h"><b>${esc(qid)}</b><span class="badge">${esc(q.type)}</span></div><div class="ins">${esc(q.instructions)}</div><div class="ans"><span class="v">${esc(head)}</span><span class="muted small">confidence ${pct(a.confidence)}</span></div>${body}</div>`;
    }).join('');
  } finally { $('#tRun').disabled = false; }
});

// ------------------------------------------------------------ boot
guard(async () => { await loadInfo(); await loadRecords(); await refreshRuns(); await refreshTrain(); });
setInterval(() => { if (TAB === 'train' || lastState === 'running') guard(refreshTrain); }, 3000);
setInterval(() => { if (TAB === 'train') guard(async () => { const d = await api('/api/info'); INFO.gpus = d.gpus; $('#gpu').textContent = d.gpus.map(x => `${x.name} ${Math.round(x.mem_used / 1024 * 10) / 10}/${Math.round(x.mem_total / 1024)} GB ${x.util}%`).join(' · ') || $('#gpu').textContent; }); }, 15000);
</script>
</body>
</html>
''')
print("wrote", os.path.join(APP_DIR, 'laya_studio_ui.html'))

In [ ]:
# 3. Start the UI on port 7860 (background thread)
import os, sys, secrets, threading, importlib, requests

APP_DIR = "/kaggle/working/laya_app"
_need = ["laya_data.py", "train_laya.py", "laya_studio_server.py", "laya_studio_ui.html"]
_missing = [f for f in _need if not os.path.exists(os.path.join(APP_DIR, f))]
if _missing:
    raise RuntimeError("App files missing from %s: %s\nRun the four 'Writes ... into APP_DIR' cells above "
                       "first (or use Run All)." % (APP_DIR, ", ".join(_missing)))
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)

def kaggle_secret(name):
    """Read a Kaggle secret; returns None if it doesn't exist or isn't attached to this notebook."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name) or None
    except Exception:
        return None

PORT = 7860
WORK_DIR = "/kaggle/working/laya_studio"
PASSWORD = kaggle_secret("LAYA_UI_PASSWORD")
if not PASSWORD:
    PASSWORD = secrets.token_urlsafe(9)
    print("No LAYA_UI_PASSWORD secret - using this one for this session:", PASSWORD)
HF_TOKEN = kaggle_secret("HF_TOKEN")   # optional: only needed for "Push to HF"

import laya_data, laya_studio_server
if "S" in globals():
    S.stop()                                  # release port 7860 before restarting
    if not S.training_running():              # reload edits, but never orphan a running training job
        importlib.reload(laya_data)
        S = importlib.reload(laya_studio_server)
else:
    S = laya_studio_server
S.start(WORK_DIR, port=PORT, password=PASSWORD, hf_token=HF_TOKEN)

for host in ("127.0.0.1", "[::1]"):
    try:
        print(host, "->", requests.get(f"http://{host}:{PORT}/health", auth=("user", PASSWORD), timeout=3).json())
    except Exception as e:
        print(host, "-> not reachable:", type(e).__name__)
print("UI login: any username, password as above. Hugging Face push:", "enabled" if HF_TOKEN else "off (no HF_TOKEN)")

# Warm the cache with the default base model so the first test/training run starts faster
def _prefetch():
    try:
        from huggingface_hub import snapshot_download
        snapshot_download("convaiinnovations/laya",
                          allow_patterns=["rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*"])
        print("Base model convaiinnovations/laya cached.")
    except Exception as e:
        print("Prefetch skipped:", e)
threading.Thread(target=_prefetch, daemon=True).start()

In [ ]:
# 4. Start this notebook's Cloudflare Tunnel in the background
import subprocess, time

TOKEN = kaggle_secret("LAYA_TUNNEL_TOKEN")
assert TOKEN, "Add the new tunnel's token as a Kaggle secret named LAYA_TUNNEL_TOKEN"

if "tunnel_proc" in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()

tunnel_log = open("/kaggle/working/cloudflared.log", "a")
tunnel_proc = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "run"],
                               stdout=tunnel_log, stderr=subprocess.STDOUT,
                               env={**os.environ, "TUNNEL_TOKEN": TOKEN})   # token never printed

for _ in range(30):
    time.sleep(1)
    if "Registered tunnel connection" in open("/kaggle/working/cloudflared.log").read():
        print("Tunnel connected. Open the public hostname you routed to http://localhost:7860")
        break
else:
    print("Tunnel not registered yet - last log lines:")
    print("".join(open("/kaggle/working/cloudflared.log").readlines()[-15:]))

In [ ]:
# 5. Keep-alive monitor. Leave it running; interrupt to stop watching (the UI and tunnel keep running).
import datetime
while True:
    ui = "up"
    try:
        requests.get(f"http://127.0.0.1:{PORT}/health", auth=("user", PASSWORD), timeout=3)
    except Exception:
        ui = "DOWN"
    tr = S.train_status()
    line = f"{datetime.datetime.now():%H:%M:%S} ui={ui} tunnel={'up' if tunnel_proc.poll() is None else 'DOWN'} training={tr.get('state')}"
    if tr.get("state") == "running":
        line += f" {tr.get('phase')} step {tr.get('step', 0)}/{tr.get('total_steps', '?')} loss {tr.get('loss', '-')}"
    print(line, flush=True)
    time.sleep(60)